## 🇺🇦 Wartime civilian harms and adolescent trauma in Ukraine: multilevel and spatial analyses leveraging geolinked OSINT data

**WIP - NOT FOR DISTRIBUTION**

**Preregistration and STROBE checklist in progress**

⛏️ `uls_scratchpad.ipynb`<br>
Simone J. Skeen x Claude Code (07-30-2026)

1. [Prepare](#1-prepare)<br>
  1a. [Define utility functions: geocoding helpers](#1a-define-utility-functions-geocoding-helpers)
2. [Import + transform: Bellingcat OSINT Civilian Harm in Ukraine](#2-import--transform-bellingcat-osint-civilian-harm-in-ukraine)
3. [Import + transform: Uppsala Conflict Data Program](#3-import--transform-uppsala-conflict-data-program)
4. [Geocode + aggregate: Bellingcat](#4-geocode--aggregate-bellingcat)<br>
  4a. [lat/long $\rightarrow$ postcode](#4a-latlong--postcode)<br>
  4b. [lat/long $\rightarrow$ _raion_](#4b-latlong--raion)
5. [Geocode + aggregate: UCDP](#5-geocode-ucdp)<br>
  5a. [lat/long $\rightarrow$ postcode](#5a-latlong--postcode)<br>
  5b. [lat/long $\rightarrow$ _raion_](#5b-latlong--raion)

### 1. Prepare
Imports requisite packages; customizes outputs; defines configuration flags.
***
**Dependencies:** Install via `pip install -r requirements.txt` from project root before running.

**Configuration:** Set `TESTING_MODE` = `True` for faster geocode testing with $n$ = 100 subset of civilian harm events; set `VALIDATION_MODE` = `True` to run validation + debug cells.

#### _Installations_

In [ ]:
%%capture

%pip install -r ../../requirements.txt

# Cell ID: b0b093cc

#### _Imports_

In [ ]:
# Standard library
import json
import os
import re
import sys
import urllib.request
import warnings
import zipfile
from datetime import datetime
from pathlib import Path
from time import sleep

# Add src to path for local imports
sys.path.insert(0, str(Path.cwd().parent))

# Third-party
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from dotenv import load_dotenv
from geopy.geocoders import Nominatim
from IPython.core.interactiveshell import InteractiveShell
from tqdm.notebook import tqdm

# Local: custom lower-level geographic mappings and UA → EN transliterations
from mappings import ADMIN_UNIT_TO_OBLAST, RAION_UA_TO_EN

# Cell ID: e690ca1a

#### _Configuration flags_

Resetting these `TRUE` vs. `FALSE` flags control the notebook execution behavior, allowing faster iterations for testing on an $n$ = 100 subset of civilian-harm instances during development.

In [ ]:
TESTING_MODE = True     ### Set True to run on n=100 subset for faster testing
VALIDATION_MODE = False  ### Set True to run validation/debug cells

# Cell ID: vm8zvul1qda

#### _Environmental variables & output preferences_

Imports ungainly and/or private variables from a `.env` ("environmental") file that is not committed to GitHub. Configures notebook output preferences, e.g. displaying all table columns rather than the default `...` truncation.

In [ ]:
# Env variables
load_dotenv()
BELLINGCAT_API_URL = os.getenv('BELLINGCAT_API_URL')
UCDP_GED_URL = os.getenv('UCDP_GED_URL')

# Output preferences
InteractiveShell.ast_node_interactivity = 'all'

pd.options.mode.copy_on_write = True
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

for category in (FutureWarning, UserWarning):
    warnings.simplefilter(action='ignore', category=category)

# Cell ID: 60998977

#### _Project directory structure (non-executable)_

Documents expected sub/directory structure as inline ASCII documentation. Created with [tree.nathanfriend.com](https://tree.nathanfriend.com/).

In [ ]:
%%script false --no-raise-error

# Project directory structure
.
└── civilian-trauma/
    ├── config
    ├── data/
    │   ├── raw/
    │   │   ├── level_1
    │   │   └── level_2
    │   └── processed
    ├── src/
    │   └── notebooks
    └── outputs/
        └── figures

# Cell ID: 6bf0d0e1

#### _Inputs subdirectories setup_

Ensures working directory is set to project root; creates expected subdirectories as needed; defines path constants (e.g. `DATA_RAW`) for single-source configuration and parsimonious handling.

In [ ]:
# Set working directory to project root; define data paths
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('../..')
elif os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')

# Inputs subdirectories
DATA_RAW = 'data/raw'
DATA_PROC = 'data/processed'
DATA_LVL1 = f'{DATA_RAW}/level_1'  ### ULS child-adolescent survey data; not for public use
DATA_LVL2 = f'{DATA_RAW}/level_2'  ### Bellingcat geotagged OSINT data

# Ensure directories exist
for path in [DATA_RAW, DATA_PROC, DATA_LVL1, DATA_LVL2]:
    os.makedirs(path, exist_ok=True)

# Cell ID: b56adf0b

#### _Define data condensation & validation constants_

Configures constants relied upon throughout the pipeline, e.g. `SURVEY_STATE_DATE`, the date of the earliest individual ULS Wave 2 observations $y_i$ and endpoint for OSINT event-level data aggregation $x_j$.

In [ ]:
# Bellingcat data source
BELLINGCAT_CSV = 'ukr-civharm-2026-01-09.csv'

# ULS merge params
SURVEY_START_DATE = '2025-04-08' ### earliest observation in ULS Wave 2 survey
DATE_FORMAT_INPUT = '%m/%d/%Y'   ### format in source data (if .CSV fallback)
DATE_FORMAT_ISO = '%Y-%m-%d'     ### ISO 8601 for internal use

# Ukrposhta postcode directory (data.gov.ua)
POSTCODE_7Z = 'zvit-dlia-miu-perelik-poshtovikh-indeksiv-ta-viddilen_08-08-2025-csv.7z'

# Geocoding
NOMINATIM_USER_AGENT = 'ukraine_postcode_geocoder'
NOMINATIM_DELAY_SEC = 1  ### 1-second delay; ensures rate limit compliance

# Cell ID: fa94b3fd

### 1a. Define utility functions: geocoding helpers
Defines two reusable reverse geocoding functions for the pipeline:

| Function | Purpose | API |
|----------|---------|-----|
| `get_postcode()` | Lat/lon → Ukrainian postal code | geopy/Nominatim |
| `raion_from_point_nominatim()` | Lat/lon → Ukrainian _raion_ (district) | Nominatim REST API |

Both functions are rate-limited to 1 request/second per Nominatim usage policy.

In [ ]:
# Cell ID: tq57p152axb
# ------------------------------------------------------------------------------
# UTILITY FUNCTIONS: GEOCODING HELPERS
# Input: Latitude/longitude coordinates; Nominatim API parameters.
# Function: Defines two core reverse geocoding functions used throughout the
#           pipeline. get_postcode() retrieves Ukrainian postal codes from
#           coordinates via geopy. raion_from_point_nominatim() retrieves
#           Ukrainian raion (district) names via direct Nominatim API calls.
#           Both functions include error handling and rate limiting compliance.
# Output: Functions available for use in Bellingcat and UCDP geocoding cells.
# ------------------------------------------------------------------------------

# Initialize geolocator for postcode lookups
geolocator = Nominatim(user_agent=NOMINATIM_USER_AGENT)


def get_postcode(lat, lon):
    """
    Reverse geocode latitude/longitude to get Ukrainian postcode.
    Returns None if postcode not found.
    """
    try:
        location = geolocator.reverse(f"{lat}, {lon}", language='en')
        if location and location.raw.get('address'):
            return location.raw['address'].get('postcode')
        return None
    except Exception as e:
        print(f"Error geocoding ({lat}, {lon}): {e}")
        return None


def raion_from_point_nominatim(lat, lon, user_agent, email=None):
    """
    Reverse geocode lat/lon to Ukrainian raion via OSM Nominatim.
    Returns raion name from 'district' field, or None if not found.
    """
    params = {
        'lat': lat,
        'lon': lon,
        'format': 'jsonv2',
        'addressdetails': 1,
    }
    headers = {'User-Agent': user_agent}
    if email:
        params['email'] = email
    
    try:
        resp = requests.get(
            'https://nominatim.openstreetmap.org/reverse',
            params=params,
            headers=headers,
            timeout=10,
        )
        resp.raise_for_status()
        data = resp.json()
        sleep(NOMINATIM_DELAY_SEC)  # respect rate limit
        return data.get('address', {}).get('district')
    except Exception as e:
        print(f"Error geocoding ({lat}, {lon}): {e}")
        return None

### 2. Import + transform: Bellingcat OSINT Civilian Harm in Ukraine
Imports, cleans, describes level-2 aggregate conflict data acquired from [Bellingcat's TimeMap instance for Civilian Harm in Ukraine](https://github.com/bellingcat/ukraine-timemap) via API endpoint (cf. `.env`). `fetch_bellingcat_json()` returns JSON list of event dictionaries. `convert_to_csv_format()` converts JSON to CSV (human-verified by SJS 07-28-2026).

In [ ]:
# Fetch routinely updated .JSON from Bellingcat API endpoint

### NOTE 7/29: `d_api.csv` & `d_dl.csv` are for visual inspection/human verification and can be deleted for prod

### docs: https://github.com/bellingcat/ukraine-timemap

def fetch_bellingcat_json(url):
    """
    Fetches Bellingcat civilian harm data from API endpoint.
    Returns list of event dictionaries.
    """
    with urllib.request.urlopen(url) as response:
        data = json.loads(response.read().decode('utf-8'))
    return data

def convert_to_csv_format(events):
    """
    Converts Bellingcat API JSON to CSV format matching ukr-civharm-*.csv structure.
    
    JSON format: id, date (YYYY-MM-DD), latitude, longitude, location, 
                 description, sources (array), impact (array), weapon_system (array)
    CSV format:  id, date (MM/DD/YYYY), latitude, longitude, location,
                 description, sources (comma-sep), associations (formatted string)
    """
    rows = []
    for event in events:
        # Convert date: YYYY-MM-DD → MM/DD/YYYY
        date_iso = event.get('date', '')
        try:
            date_obj = datetime.strptime(date_iso, '%Y-%m-%d')
            date_formatted = date_obj.strftime('%m/%d/%Y')
        except ValueError:
            date_formatted = date_iso
        
        # Join sources array
        sources = event.get('sources', [])
        sources_str = ','.join(sources) if sources else ''
        
        # Build `associations` string from `impact` & `weapon_system`
        associations_parts = []
        for impact in event.get('impact', []):
            associations_parts.append(f'Type of area affected={impact}')
        for weapon in event.get('weapon_system', []):
            associations_parts.append(f'Weapon System={weapon}')
        associations_str = ','.join(associations_parts) if associations_parts else ''
        
        rows.append({
            'id': event.get('id', ''),
            'date': date_formatted,
            'latitude': event.get('latitude', ''),
            'longitude': event.get('longitude', ''),
            'location': (event.get('location') or '').strip(),
            'description': (event.get('description') or '').strip(),
            'sources': sources_str,
            'associations': associations_str,
        })
    
    return pd.DataFrame(rows)

# Fetch
print(f"Fetching data from Bellingcat API...")
d_lvl2_bcat_raw_json = fetch_bellingcat_json(BELLINGCAT_API_URL)

# Save raw .JSON 
json_path = f"{DATA_LVL2}/ukr-civharm-{datetime.now().strftime('%Y-%m-%d')}.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(d_lvl2_bcat_raw_json, f, ensure_ascii=False, indent=2)
print(f"Raw JSON saved to: {json_path}")

# Convert & save to .CSV
d_lvl2_bcat_raw_csv = convert_to_csv_format(d_lvl2_bcat_raw_json)

# Sort by date (earliest first)
d_lvl2_bcat_raw_csv['_date_sort'] = pd.to_datetime(d_lvl2_bcat_raw_csv['date'], format='%m/%d/%Y', errors='coerce')
d_lvl2_bcat_raw_csv = d_lvl2_bcat_raw_csv.sort_values('_date_sort').drop(columns=['_date_sort']).reset_index(drop=True)

csv_path = f"{DATA_LVL2}/ukr-civharm-{datetime.now().strftime('%Y-%m-%d')}.csv"
d_lvl2_bcat_raw_csv.to_csv(csv_path, index=False)

print(f"Fetched {len(d_lvl2_bcat_raw_csv):,} events")
print(f"Saved to: {csv_path}")
d_lvl2_bcat_raw_csv.head(3)

# Cell ID: 1khtuwbrxcdh

#### _Bellingcat raw data housekeeping_

Indexes and condenses the "raw" event-level API data, applies `SURVEY_START_DATE` datetime restriction. 

In [ ]:
# Dupe raw for processing
d_lvl2_bcat = d_lvl2_bcat_raw_csv.copy()

# Add ascending numerical index
d_lvl2_bcat['index'] = range(len(d_lvl2_bcat))
d_lvl2_bcat = d_lvl2_bcat.set_index('index')

# Drop imprecise OS location col
d_lvl2_bcat = d_lvl2_bcat.drop(
    'location', 
    axis = 1, 
    errors = 'ignore',
    )

# Restrict to obs on or before ULS start date
d_lvl2_bcat['date'] = pd.to_datetime(
    d_lvl2_bcat['date'], 
    format = DATE_FORMAT_INPUT,
    errors = 'coerce',    
    )

uls_startdate = pd.to_datetime(SURVEY_START_DATE)
d_lvl2_bcat = d_lvl2_bcat[d_lvl2_bcat['date'] <= uls_startdate]

# Inspect & verify
d_lvl2_bcat.shape
d_lvl2_bcat.info()
d_lvl2_bcat.head(2)
d_lvl2_bcat.tail(2)

# Cell ID: a1cadbe8

#### _Bellingcat processed data dummy coding_

Disaggregates original Bellingcat OSINT comma-separated `associations` strings, dummy codes indicators for type of civilian infrastructrual damage and observed weapons systems. 

In [ ]:
# Dummy code: area type affected & weapon system

# === TYPE OF AREA AFFECTED ===
area_types = {
    'a00': 'Administrative',           
    'a01': 'Commercial',
    'a02': 'Cultural',
    'a03': 'Food/Food Infrastructure',
    'a04': 'Healthcare',
    'a05': 'Humanitarian',
    'a06': 'Industrial',
    'a07': 'Military',
    'a08': 'Religious',
    'a09': 'Residential',
    'a10': 'Roads/Highways/Transport',
    'a11': 'School or childcare',
    'undefined': 'Undefined',
    }

for var, label in area_types.items():
    d_lvl2_bcat[var] = d_lvl2_bcat['associations'].str.contains(
        rf'Type of area affected={re.escape(label)}',
        case=False,
        na=False,
        regex=True,
    ).astype(int)

# === WEAPON SYSTEM ===
weapon_systems = {
    'w00': 'Air strike',
    'w01': 'Anti-air missile',
    'w02': 'Ballistic missile',
    'w03': 'Cluster munitions',
    'w04': 'Cruise missile',
    'w05': 'HE artillery inc mortars',
    'w06': 'HE rocket artillery',
    'w07': 'HE tube artillery',
    'w08': 'Incendiary munitions',
    'w09': 'Land mines',
    'w10': 'Loitering munition',
    'w11': 'Small arms',
    'w12': 'Thermobaric munition',
    'w13': 'Vehicle mounted weapon',
    'unknown': 'Unknown',
    'none': 'None',
    }

for var, label in weapon_systems.items():
    d_lvl2_bcat[var] = d_lvl2_bcat['associations'].str.contains(
        rf'Weapon System={re.escape(label)}',
        case=False,
        na=False,
        regex=True,
    ).astype(int)

# Verify counts
print("=== TYPE OF AREA AFFECTED ===")
for var, label in area_types.items():
    print(f"  {var} ({label}): {d_lvl2_bcat[var].sum()}")

print("\n=== WEAPON SYSTEM ===")
for var, label in weapon_systems.items():
    print(f"  {var} ({label}): {d_lvl2_bcat[var].sum()}")

# Cell ID: gsgy8ry44j4

In [ ]:
# Inspect dummy-coded df
d_lvl2_bcat.head(5)

# Cell ID: 26c6ba36

### 3. Import + transform: Uppsala Conflict Data Program
Imports, cleans, describes level-2 [Uppsala Conflict Data Program](https://ucdp.uu.se/) (UCDP) Georeferenced Event Dataset global archive; extracts CSV; loads data into a DataFrame; defines `fetch_ucdp_ged()` helper function for download and extraction.

In [ ]:
# Fetch UCDP Georeferenced Event Dataset Global

### docs: https://ucdp.uu.se/downloads/index.html
### codebook: https://ucdp.uu.se/downloads/ged/ged261.pdf

def fetch_ucdp_ged(url, output_dir):
    """
    Downloads and extracts UCDP GED CSV from zip archive.
    Returns path to extracted CSV file.
    """
    zip_filename = url.split('/')[-1]
    zip_path = f"{output_dir}/{zip_filename}"
    
    # Download zip file
    print(f"Downloading {zip_filename}...")
    urllib.request.urlretrieve(url, zip_path)
    print(f"Downloaded to: {zip_path}")
    
    # Extract CSV from zip
    print(f"Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        csv_files = [f for f in z.namelist() if f.endswith('.csv')]
        if not csv_files:
            raise ValueError("No CSV file found in archive")
        
        csv_filename = csv_files[0]
        z.extract(csv_filename, output_dir)
        csv_path = f"{output_dir}/{csv_filename}"
    
    # Remove zip file after extraction
    os.remove(zip_path)
    print(f"Extracted to: {csv_path}")
    
    return csv_path

# Fetch UCDP GED
ucdp_csv_path = fetch_ucdp_ged(UCDP_GED_URL, DATA_LVL2)

# Load and preview
d_lvl2_ucdp_raw = pd.read_csv(ucdp_csv_path, low_memory=False)
print(f"UCDP GED loaded: {len(d_lvl2_ucdp_raw):,} events (global)")
print(f"Columns: {d_lvl2_ucdp_raw.columns.tolist()}")
d_lvl2_ucdp_raw.head(3)

# Cell ID: e75de4bb

#### _UCDP raw data housekeeping_

Indexes and condenses the "raw" event-level CSV data, subsets to Russo-Ukrainian War `conflict_name`, and post-2022-02-24 datetime, applies `SURVEY_START_DATE` datetime restriction.

In [ ]:
# Dupe raw for processing
d_lvl2_ucdp = d_lvl2_ucdp_raw.copy()

# Filter to Russia-Ukraine conflict only
d_lvl2_ucdp = d_lvl2_ucdp[d_lvl2_ucdp['conflict_name'] == 'Russia - Ukraine']
print(f"Filtered to Russia-Ukraine: {len(d_lvl2_ucdp):,} events")

# Convert date_start to datetime
d_lvl2_ucdp['date_start'] = pd.to_datetime(d_lvl2_ucdp['date_start'], errors='coerce')

# Filter to invasion start (2022-02-24) through ULS survey start
invasion_start = pd.to_datetime('2022-02-24')
uls_startdate = pd.to_datetime(SURVEY_START_DATE)

d_lvl2_ucdp = d_lvl2_ucdp[
    (d_lvl2_ucdp['date_start'] >= invasion_start) & 
    (d_lvl2_ucdp['date_start'] <= uls_startdate)
]
print(f"Filtered to {invasion_start.date()} – {uls_startdate.date()}: {len(d_lvl2_ucdp):,} events")

# Sort by date ascending
d_lvl2_ucdp = d_lvl2_ucdp.sort_values('date_start').reset_index(drop=True)

# Inspect & verify
d_lvl2_ucdp.shape
d_lvl2_ucdp.info()
d_lvl2_ucdp.head(3)

# Cell ID: sg7h5uexx6p

In [ ]:
# Data condensation: keep only relevant columns
cols_keep = [
    'conflict_name', 'source_article', 'source_office',
    'source_headline', 'source_original', 'where_prec',
    'where_description', 'adm_1', 'adm_2',
    'latitude', 'longitude', 'geom_wkt',
    'priogrid_gid', 'country', 'event_clarity',
    'date_prec', 'date_start', 'date_end',
    'deaths_a', 'deaths_b', 'deaths_civilians',
    'deaths_unknown', 'best', 'high',
    ]

d_lvl2_ucdp = d_lvl2_ucdp[cols_keep]

print(f"Condensed to {len(cols_keep)} columns")
d_lvl2_ucdp.info()
d_lvl2_ucdp.head(3)

# Cell ID: 9f3b8uuw4x

### 4. Geocode + aggregate: Bellingcat
#### 4a. lat/long $\rightarrow$ postcode
Reverse geocodes latitudinal and longitudinal coordinates to Ukrainian postcodes via Nominatim using `get_postcode()` (defined in §1b). Postcodes enable 2-digit admin unit extraction for oblast mapping and cross-validation of _raion_ geocoding.

In [ ]:
# Restrict to n=100 for geocoding tests (set TESTING_MODE = True in Configuration)
if TESTING_MODE:
    d_lvl2_bcat = d_lvl2_bcat.iloc[:100]
    print(f"TESTING_MODE: Restricted to {len(d_lvl2_bcat)} rows")
    d_lvl2_bcat.info()

# Cell ID: d5a7124d

In [ ]:
# Apply `get_postcode()` to each row with rate-limited delay
# (function defined in Cell ID: tq57p152axb)

postcodes = []

for idx, row in tqdm(d_lvl2_bcat.iterrows(), total=len(d_lvl2_bcat), desc="Geocoding Bellingcat postcodes"):
    lat = row['latitude']
    lon = row['longitude']
    
    postcode = get_postcode(lat, lon)
    postcodes.append(postcode)
    
    sleep(NOMINATIM_DELAY_SEC)

d_lvl2_bcat['postcode'] = postcodes

print(f"\nGeocoding complete!")
print(f"Postcodes found: {d_lvl2_bcat['postcode'].notna().sum()}/{len(d_lvl2_bcat)}")
print(f"\nSample results:")
print(d_lvl2_bcat[['latitude', 'longitude', 'postcode']].head(10))

# Cell ID: bafc051c

#### _Map postcode to_ oblast
Extracts 2-digit leading `admin_unit` from Ukrainian postcode to assign _oblast_; calls `ADMIN_UNIT_TO_OBLAST`
mappings dictionary from `mappings.py`.

In [ ]:
# Extract first 2 digits of postcode as `admin_unit` code
d_lvl2_bcat['admin_unit'] = d_lvl2_bcat['postcode'].astype(str).str[:2]

# Replace 'na' (from NaN conversion) with actual NaN
d_lvl2_bcat.loc[d_lvl2_bcat['admin_unit'] == 'na', 'admin_unit'] = np.nan

# Map `admin_unit` → `oblast`
### Uses ADMIN_UNIT_TO_OBLAST from mappings.py
### Source: Ukrposhta postal code system: https://en.wikipedia.org/wiki/Postal_codes_in_Ukraine
d_lvl2_bcat['oblast'] = d_lvl2_bcat['admin_unit'].map(ADMIN_UNIT_TO_OBLAST)

# Report coverage
print(f"Unique admin units: {d_lvl2_bcat['admin_unit'].dropna().nunique()}")
print(f"Mapped oblasts: {d_lvl2_bcat['oblast'].notna().sum()}/{len(d_lvl2_bcat)}")
print(f"Unmapped admin units: {d_lvl2_bcat[d_lvl2_bcat['oblast'].isna()]['admin_unit'].unique()}")
print(f"\nOblast distribution:")
d_lvl2_bcat['oblast'].value_counts()

# Cell ID: x92lyfbyqkd

### 4b. lat/long $\rightarrow$ _raion_
Reverse geocodes coordinates to Ukrainian _raion_ (district) names via `raion_from_point_nominatim()` (defined in §1b). Uses Nominatim's `district` field to return _raion_ names in Ukrainian Cyrillic.

**Note:** Ukrposhta postcode lookup (optional validation below) will not return Russian-occupied districts:

|Prefix|Region|Status|
|------|------|------|
|83___|Donetsk city|`<Rus-occupied>`|
|91___|Luhansk city|`<Rus-occupied>`|
|94___|Luhansk oblast|`<Rus-occupied>`|
|95-99|Crimea/Sevastopol|`<Rus-occupied>`|

In [ ]:
# Apply `raion_from_point_nominatim()` to each row with progress bar
# (function defined in Cell ID: tq57p152axb)

raions_nominatim = []

for idx, row in tqdm(d_lvl2_bcat.iterrows(), total=len(d_lvl2_bcat), desc="Geocoding Bellingcat raions"):
    raion = raion_from_point_nominatim(
        row['latitude'], 
        row['longitude'], 
        user_agent=NOMINATIM_USER_AGENT,
    )
    raions_nominatim.append(raion)

d_lvl2_bcat['raion_nominatim_ua'] = raions_nominatim

print(f"\nRaion geocoding complete!")
print(f"Raions found: {d_lvl2_bcat['raion_nominatim_ua'].notna().sum()}/{len(d_lvl2_bcat)}")
print(f"\nSample results:")
d_lvl2_bcat[['latitude', 'longitude', 'raion_nominatim_ua']].head(10)

# Cell ID: e6cb88be

#### _Transliterate_ raion _names to English_
Transliterates Ukrainian _raion_ names to English equivalents using the `RAION_UA_TO_EN` lookup dictionary (based on post-2020 administrative reform with 136 _raions_). Reports translation coverage; identifies any unmapped Ukrainian _raion_ names that need to be added to `mappings.py` for complete coverage.

In [ ]:
# Map Ukrainian raion names → English translations
### Uses RAION_UA_TO_EN from `mappings.py`
### Source: https://en.wikipedia.org/wiki/Raions_of_Ukraine (post-2020 reform: 136 raions)

d_lvl2_bcat['raion_nominatim_en'] = d_lvl2_bcat['raion_nominatim_ua'].map(RAION_UA_TO_EN)

# Report coverage
mapped = d_lvl2_bcat['raion_nominatim_en'].notna().sum()
total_with_ua = d_lvl2_bcat['raion_nominatim_ua'].notna().sum()
print(f"Mapped to English: {mapped}/{total_with_ua}")

# Show any unmapped Ukrainian raions for dictionary updates
unmapped = d_lvl2_bcat[d_lvl2_bcat['raion_nominatim_ua'].notna() & d_lvl2_bcat['raion_nominatim_en'].isna()]['raion_nominatim_ua'].unique()
if len(unmapped) > 0:
    print(f"\nUnmapped raions (add to mappings.py):")
    for r in unmapped:
        print(f"    '{r}': '',")

print(f"\nSample results:")
d_lvl2_bcat[['raion_nominatim_ua', 'raion_nominatim_en']].head(10)

# Cell ID: ta4wg5hmu0s

#### _Map postcodes $\rightarrow$ Ukrainian_ raions
Leverages open-source Ukrainian administrative data to cross-validate lat/long to _raion_ encodings using lookup table. 

In [ ]:
# Extract postcode directory from .7z archive
import py7zr

POSTCODE_7Z_PATH = f'{DATA_RAW}/{POSTCODE_7Z}'

# Extract if .7z exists
if os.path.exists(POSTCODE_7Z_PATH):
    # Check if already extracted by looking for any CSV with Ukrainian postal keywords
    existing_csvs = [f for f in os.listdir(DATA_RAW) if f.endswith('.csv') and 'індекс' in f.lower()]
    
    if not existing_csvs:
        print(f"Extracting {POSTCODE_7Z_PATH}...")
        with py7zr.SevenZipFile(POSTCODE_7Z_PATH, mode='r') as archive:
            archive.extractall(path=DATA_RAW)
        print(f"Extracted to: {DATA_RAW}/")
        existing_csvs = [f for f in os.listdir(DATA_RAW) if f.endswith('.csv') and 'індекс' in f.lower()]
    
    # Set path to the extracted CSV
    if existing_csvs:
        POSTCODE_DIR_PATH = f'{DATA_RAW}/{existing_csvs[0]}'
        print(f"Postcode directory: {POSTCODE_DIR_PATH}")
    else:
        # Fallback: find any newly created CSV
        all_csvs = [f for f in os.listdir(DATA_RAW) if f.endswith('.csv')]
        print(f"CSV files found: {all_csvs}")
        if all_csvs:
            POSTCODE_DIR_PATH = f'{DATA_RAW}/{all_csvs[0]}'
            print(f"Using: {POSTCODE_DIR_PATH}")
else:
    print(f"Archive not found: {POSTCODE_7Z_PATH}")
    print("Download from: https://data.gov.ua/dataset/post-index-and-braches")
    POSTCODE_DIR_PATH = None

# Cell ID: nrh6dku56f

In [ ]:
# Map postcodes -> Ukrainian raions via Ministry of Community and Territorial Development of Ukraine open data
### Source: ukraine_raion_lookup.py (option 3)
### Data: https://data.gov.ua/dataset/post-index-and-braches
### Note: Column headers are in Ukrainian and may vary between releases; inspect after download

def load_postcode_directory(csv_path):
    """
    Load the data.gov.ua "post-index-and-braches" CSV.
    Tries multiple encodings common for Ukrainian government data.
    Uses semicolon delimiter (European CSV format).
    """
    encodings = ['cp1251', 'windows-1251', 'utf-8', 'iso-8859-5', 'utf-16']
    
    for encoding in encodings:
        try:
            df = pd.read_csv(
                csv_path, 
                encoding=encoding, 
                dtype=str,
                sep=';',            ### European .CSV uses semicolon
                on_bad_lines='skip' ### Skips malformed rows
            )
            print(f"Successfully loaded with encoding: {encoding}")
            return df
        except (UnicodeDecodeError, UnicodeError):
            continue
    
    raise ValueError(f"Could not decode {csv_path} with any known encoding")

def raion_from_postcode(postcode, directory, postcode_col, raion_col):
    """
    Direct table lookup for postcode -> raion.
    Ukrainian postal codes do NOT cleanly encode raion in digit positions;
    use this authoritative directory rather than parsing the string.
    """
    row = directory.loc[directory[postcode_col] == str(postcode)]
    if row.empty:
        return None
    return row.iloc[0][raion_col]

# Load directory and inspect columns
try:
    postcode_dir = load_postcode_directory(POSTCODE_DIR_PATH)
    print(f"Postcode directory loaded: {len(postcode_dir):,} entries")
    print(f"Columns: {postcode_dir.columns.tolist()}")
    
    # Use English column names from the file
    # Adjust these if your file has different column names
    POSTCODE_COL = 'Postindex VPZ'    ### postcode column
    RAION_COL = 'Distinct (Rayon)'    ### raion column (note: "Distinct" is likely a typo for "District")
    
    # Verify columns exist
    if POSTCODE_COL not in postcode_dir.columns or RAION_COL not in postcode_dir.columns:
        print(f"\nWARNING: Expected columns not found!")
        print(f"Looking for: '{POSTCODE_COL}', '{RAION_COL}'")
        print(f"Available: {postcode_dir.columns.tolist()}")
    else:
        # Apply lookup to dataframe
        d_lvl2_bcat['raion_postcode'] = d_lvl2_bcat['postcode'].apply(
            lambda pc: raion_from_postcode(pc, postcode_dir, POSTCODE_COL, RAION_COL)
        )
        
        # Replace None with <Rus-occupied> (postcodes in occupied territories not in Ukrposhta directory)
        d_lvl2_bcat['raion_postcode'] = d_lvl2_bcat['raion_postcode'].fillna('<Rus-occupied>')
        
        print(f"\nRaions found via postcode: {(d_lvl2_bcat['raion_postcode'] != '<Rus-occupied>').sum()}/{len(d_lvl2_bcat)}")
        print(f"Rus-occupied: {(d_lvl2_bcat['raion_postcode'] == '<Rus-occupied>').sum()}/{len(d_lvl2_bcat)}")
        print(f"\nSample results:")
        d_lvl2_bcat[['postcode', 'raion_postcode']].head(10)
    
except FileNotFoundError:
    print(f"Postcode directory not found at: {POSTCODE_DIR_PATH}")
    print("Download from: https://data.gov.ua/dataset/post-index-and-braches")
    print("Save as 'postindex.7z' in data/raw/ and run extraction cell above")

# Cell ID: a5a71bd6

In [ ]:
# DEBUG: Investigate postcode lookup misses (set VALIDATION_MODE = True in Configuration)
if VALIDATION_MODE:
    test_postcode = '83054'
    print(f"Investigating postcode: {test_postcode}\n")

    # Check both postcode columns in directory
    postcode_cols = ['Postindex VPZ', 'Postindex Locality']
    for col in postcode_cols:
        if col in postcode_dir.columns:
            exact = postcode_dir[postcode_dir[col] == test_postcode]
            partial = postcode_dir[postcode_dir[col].str.contains(test_postcode, na=False)]
            print(f"'{col}':")
            print(f"  Exact match: {len(exact)} rows")
            print(f"  Partial match: {len(partial)} rows")
            print(f"  Sample values: {postcode_dir[col].dropna().unique()[:10].tolist()}\n")

    # Check what postcodes ARE in directory for Donetsk oblast (83-87)
    donetsk_postcodes = postcode_dir[postcode_dir['Postindex VPZ'].str.startswith('83', na=False)]
    print(f"Donetsk (83xxx) postcodes in directory: {len(donetsk_postcodes)}")
    if len(donetsk_postcodes) > 0:
        print(f"Sample: {donetsk_postcodes['Postindex VPZ'].unique()[:10].tolist()}")

    # Check Luhansk (91-94)
    luhansk_postcodes = postcode_dir[postcode_dir['Postindex VPZ'].str.startswith('92', na=False)]
    print(f"\nLuhansk (92xxx) postcodes in directory: {len(luhansk_postcodes)}")
    if len(luhansk_postcodes) > 0:
        print(f"Sample: {luhansk_postcodes['Postindex VPZ'].unique()[:10].tolist()}")

    # Summary: which oblasts are missing?
    print(f"\nOblast prefixes in directory:")
    postcode_dir['prefix'] = postcode_dir['Postindex VPZ'].str[:2]
    print(postcode_dir['prefix'].value_counts().sort_index())

# Cell ID: 20tnlz3i1y2

In [ ]:
# Inspect df
d_lvl2_bcat.head(5)

# Cell ID: 46aa286d

#### _Cross-validate_ raion _mappings_

Flags mismatches (if any) between lat/long and postcode geocodings.

In [ ]:
# Validation: Compare raion mappings from both methods (set VALIDATION_MODE = True in Configuration)
if VALIDATION_MODE:
    # Check where both methods returned a result
    both_valid = d_lvl2_bcat['raion_nominatim_ua'].notna() & (d_lvl2_bcat['raion_postcode'] != '<Rus-occupied>')
    n_both = both_valid.sum()

    print(f"Rows with both raion values: {n_both}/{len(d_lvl2_bcat)}")

    if n_both > 0:
        # Compare results (case-insensitive, strip whitespace)
        d_lvl2_bcat['raion_match'] = d_lvl2_bcat.apply(
            lambda row: (
                str(row['raion_nominatim_ua']).lower().strip() == 
                str(row['raion_postcode']).lower().strip()
            ) if pd.notna(row['raion_nominatim_ua']) and row['raion_postcode'] != '<Rus-occupied>' else np.nan,
            axis=1
        )
        
        n_match = d_lvl2_bcat['raion_match'].sum()
        match_rate = n_match / n_both * 100 if n_both > 0 else 0
        
        print(f"Exact matches: {int(n_match)}/{n_both} ({match_rate:.1f}%)")
        
        # Show mismatches for inspection
        mismatches = d_lvl2_bcat[both_valid & (d_lvl2_bcat['raion_match'] == False)][
            ['postcode', 'latitude', 'longitude', 'raion_nominatim_ua', 'raion_postcode']
        ]
        if len(mismatches) > 0:
            print(f"\nMismatches ({len(mismatches)}):")
            mismatches.head(10)
        else:
            print("\nNo mismatches found!")

# Cell ID: c5bb5a9c

#### _Inspect Nominatim response structure_

Debug cell: queries Nominatim for a sample coordinate and displays the full address breakdown; shows all available address components (country, state, county, city, etc.) to help understand the response structure and inform parsing logic.

In [ ]:
# DEBUG: Inspect raw Nominatim response for a sample coordinate (set VALIDATION_MODE = True in Configuration)
if VALIDATION_MODE:
    import requests

    # Use first row with valid coordinates
    sample_row = d_lvl2_bcat[d_lvl2_bcat['latitude'].notna()].iloc[0]
    lat, lon = sample_row['latitude'], sample_row['longitude']

    print(f"Testing coordinates: ({lat}, {lon})")

    params = {
        'lat': lat,
        'lon': lon,
        'format': 'jsonv2',
        'addressdetails': 1,
        }
    headers = {'User-Agent': NOMINATIM_USER_AGENT}

    resp = requests.get(
        'https://nominatim.openstreetmap.org/reverse',
        params=params,
        headers=headers,
        timeout=10,
        )
    data = resp.json()

    print(f"\nFull address breakdown:")
    for key, value in data.get('address', {}).items():
        print(f"  {key}: {value}")

    print(f"\nDisplay name: {data.get('display_name', 'N/A')}")

# Cell ID: 9e478859

In [ ]:
# Export event-level data prior to aggregation
d_lvl2_bcat.to_csv(f'{DATA_PROC}/d_lvl2_bcat_event.csv', index=False)

#### _Aggregate Bellingcat event counts by_ raion

Collapses event-level Bellingcat data to _raion_-level raw counts of type of civilian infrastructrual damage and observed weapons systems.

In [ ]:
### Collapses event-level data to raion-level counts
### Uses raion_nominatim_en as primary raion identifier

# Define all dummy variables to aggregate
area_type_vars = ['a00', 'a01', 'a02', 'a03', 'a04', 'a05', 
                  'a06', 'a07', 'a08', 'a09', 'a10', 'a11', 
                  'undefined']

weapon_sys_vars = ['w00', 'w01', 'w02', 'w03', 'w04', 'w05',
                   'w06', 'w07', 'w08', 'w09', 'w10', 'w11', 
                   'w12', 'w13', 'unknown', 'none']

all_dummy_vars = area_type_vars + weapon_sys_vars

# Build aggregation dict: sum all dummy vars, count events
agg_dict = {var: 'sum' for var in all_dummy_vars}
agg_dict['id'] = 'count'  # count events per raion

# Aggregate by raion (English name)
d_lvl2_bcat_raion = d_lvl2_bcat.groupby('raion_nominatim_en', as_index=False).agg(agg_dict)

# Rename id count column
d_lvl2_bcat_raion = d_lvl2_bcat_raion.rename(columns={'id': 'n_events'})

# Add oblast mapping (most common oblast per raion)
oblast_map = d_lvl2_bcat.groupby('raion_nominatim_en')['oblast'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan
)
d_lvl2_bcat_raion['oblast'] = d_lvl2_bcat_raion['raion_nominatim_en'].map(oblast_map)

# Reorder columns: raion, oblast, n_events, area types, weapon systems
col_order = ['raion_nominatim_en', 'oblast', 'n_events'] + area_type_vars + weapon_sys_vars
d_lvl2_bcat_raion = d_lvl2_bcat_raion[col_order]

# Sort by total events (descending)
d_lvl2_bcat_raion = d_lvl2_bcat_raion.sort_values('n_events', ascending=False).reset_index(drop=True)

# Summary
print(f"Raion-level aggregation: {len(d_lvl2_bcat_raion)} raions")
print(f"Total events: {d_lvl2_bcat_raion['n_events'].sum():,}")
print(f"\nTop 10 raions by event count:")
d_lvl2_bcat_raion.head(10)

# Cell ID: 466bcf8d

In [ ]:
# Export Bellingcat raion-level estimates for 1:n merge
d_lvl2_bcat_raion.to_csv(f'{DATA_PROC}/d_lvl2_bcat_raion.csv', index=False)

# Cell ID: 21454123

#### _Aggregate Bellingcat event counts by_ oblast

Collapses event-level Bellingcat data to _oblast_-level raw counts of type of civilian infrastructrual damage and observed weapons systems.

In [ ]:
### Collapses event-level data to oblast-level counts

# Build aggregation dict: sum all dummy vars, count events
agg_dict_obl = {var: 'sum' for var in all_dummy_vars}
agg_dict_obl['id'] = 'count'

# Aggregate by oblast
d_lvl2_bcat_oblast = d_lvl2_bcat.groupby('oblast', as_index=False).agg(agg_dict_obl)

# Rename id count column
d_lvl2_bcat_oblast = d_lvl2_bcat_oblast.rename(columns={'id': 'n_events'})

# Add raion count per oblast
raion_counts = d_lvl2_bcat.groupby('oblast')['raion_nominatim_en'].nunique()
d_lvl2_bcat_oblast['n_raions'] = d_lvl2_bcat_oblast['oblast'].map(raion_counts)

# Reorder columns: oblast, n_raions, n_events, area types, weapon systems
col_order_obl = ['oblast', 'n_raions', 'n_events'] + area_type_vars + weapon_sys_vars
d_lvl2_bcat_oblast = d_lvl2_bcat_oblast[col_order_obl]

# Sort by total events (descending)
d_lvl2_bcat_oblast = d_lvl2_bcat_oblast.sort_values('n_events', ascending=False).reset_index(drop=True)

# Summary
print(f"Oblast-level aggregation: {len(d_lvl2_bcat_oblast)} oblasts")
print(f"Total events: {d_lvl2_bcat_oblast['n_events'].sum():,}")
print(f"\nOblast counts:")
d_lvl2_bcat_oblast

# Cell ID: zyvmk8t88h

In [ ]:
# Export Bellingcat oblast-level estimates for 1:n merge
d_lvl2_bcat_oblast.to_csv(f'{DATA_PROC}/d_lvl2_bcat_oblast.csv', index=False)

# Cell ID: 527c94e6

### 5. Geocode: UCDP
#### 5a. lat/long $\rightarrow$ postcode
Reverse geocodes latitudinal and longitudinal coordinates to Ukrainian postcodes via Nominatim using `get_postcode()` (defined in §1b). Postcodes enable 2-digit admin unit extraction for oblast mapping and cross-validation of _raion_ geocoding.

In [ ]:
# Restrict to n=100 for geocoding tests (set TESTING_MODE = True in Configuration)
if TESTING_MODE:
    d_lvl2_ucdp = d_lvl2_ucdp.iloc[:100]
    print(f"TESTING_MODE: Restricted to {len(d_lvl2_ucdp)} rows")
    d_lvl2_ucdp.info()

# Cell ID: 43a2244c

In [ ]:
# Convert latitude/longitude coordinates → Ukrainian postcodes via Nominatim API
# Reuses get_postcode() function defined in Utility Functions section

postcodes_ucdp = []

for idx, row in tqdm(d_lvl2_ucdp.iterrows(), total=len(d_lvl2_ucdp), desc="Geocoding UCDP postcodes"):
    lat = row['latitude']
    lon = row['longitude']
    
    postcode = get_postcode(lat, lon)
    postcodes_ucdp.append(postcode)
    
    sleep(NOMINATIM_DELAY_SEC)

d_lvl2_ucdp['postcode'] = postcodes_ucdp

print(f"\nGeocoding complete!")
print(f"Postcodes found: {d_lvl2_ucdp['postcode'].notna().sum()}/{len(d_lvl2_ucdp)}")

# Cell ID: 1770bddb

#### _Map postcode to_ oblast
Extracts 2-digit leading `admin_unit` from Ukrainian postcode to assign _oblast_; calls `ADMIN_UNIT_TO_OBLAST`
mappings dictionary from `mappings.py`.

In [ ]:
# Extract first 2 digits of postcode as `admin_unit` id
d_lvl2_ucdp['admin_unit'] = d_lvl2_ucdp['postcode'].astype(str).str[:2]

# Replace 'na' (from NaN conversion) with actual NaN
d_lvl2_ucdp.loc[d_lvl2_ucdp['admin_unit'] == 'na', 'admin_unit'] = np.nan

# Map `admin_unit` → `oblast`
d_lvl2_ucdp['oblast'] = d_lvl2_ucdp['admin_unit'].map(ADMIN_UNIT_TO_OBLAST)

# Verify mapping
print(f"Mapped oblasts: {d_lvl2_ucdp['oblast'].notna().sum()}/{len(d_lvl2_ucdp)}")
print(f"\nOblast distribution:")
d_lvl2_ucdp['oblast'].value_counts()

# Cell ID: c1251e5f

### 5b. lat/long $\rightarrow$ _raion_
Reverse geocodes coordinates to Ukrainian _raion_ (district) names via `raion_from_point_nominatim()` (defined in §1b). Uses Nominatim's `district` field to return _raion_ names in Ukrainian Cyrillic. Note contraints related to Russion-occupied districts tabulated in §4b. 

In [ ]:
# Map latitude/longitude coordinates → Ukrainian raions via Nominatim API
# Reuses raion_from_point_nominatim() function defined in Utility Functions section

raions_ucdp = []

for idx, row in tqdm(d_lvl2_ucdp.iterrows(), total=len(d_lvl2_ucdp), desc="Geocoding UCDP raions"):
    raion = raion_from_point_nominatim(
        row['latitude'],
        row['longitude'],
        user_agent=NOMINATIM_USER_AGENT,
    )
    raions_ucdp.append(raion)

d_lvl2_ucdp['raion_nominatim_ua'] = raions_ucdp

print(f"\nRaion geocoding complete!")
print(f"Raions found: {d_lvl2_ucdp['raion_nominatim_ua'].notna().sum()}/{len(d_lvl2_ucdp)}")

# Cell ID: 9fd0ac7c

#### _Transliterate_ raion _names to English_
Transliterates Ukrainian _raion_ names to English equivalents using the `RAION_UA_TO_EN` lookup dictionary (based on post-2020 administrative reform with 136 _raions_). Reports translation coverage; identifies any unmapped Ukrainian _raion_ names that need to be added to `mappings.py` for complete coverage.

In [ ]:
# Map Ukrainian raion names → English translations
d_lvl2_ucdp['raion_nominatim_en'] = d_lvl2_ucdp['raion_nominatim_ua'].map(RAION_UA_TO_EN)

# Report coverage
mapped = d_lvl2_ucdp['raion_nominatim_en'].notna().sum()
total_with_ua = d_lvl2_ucdp['raion_nominatim_ua'].notna().sum()
print(f"Mapped to English: {mapped}/{total_with_ua}")

# Show any unmapped Ukrainian raions for dictionary updates
unmapped = d_lvl2_ucdp[d_lvl2_ucdp['raion_nominatim_ua'].notna() & d_lvl2_ucdp['raion_nominatim_en'].isna()]['raion_nominatim_ua'].unique()
if len(unmapped) > 0:
    print(f"\nUnmapped raions (add to mappings.py):")
    for r in unmapped:
        print(f"    '{r}': '',")

print(f"\nSample results:")
d_lvl2_ucdp[['latitude', 'longitude', 'oblast', 'raion_nominatim_ua', 'raion_nominatim_en']].head(10)

# NOTE: UTF-8 Cyrillic encoding will not display properly in Excel with Latin-1 defaults

# Cell ID: a6924201

In [ ]:
# Export event-level data prior to aggregation
d_lvl2_ucdp.to_csv(f'{DATA_PROC}/d_lvl2_ucdp_event.csv', index=False)

# Cell ID: 197cfdcf

#### _Aggregate UCDP event counts by_ raion

Collapses event-level UCDP data to _raion_-level waw counts of Ukrainian military (`deaths_b`) and civilian deaths (`deaths_civilians`).

In [ ]:
# Aggregate UCDP at raion level

### Collapses event-level data to raion-level counts
### Uses raion_nominatim_en as primary raion identifier

# Define variables to aggregate
ucdp_agg_vars = ['deaths_b', 'deaths_civilians']

# Build aggregation dict: sum deaths, count events
agg_dict_ucdp = {var: 'sum' for var in ucdp_agg_vars}
agg_dict_ucdp['conflict_name'] = 'count'  # count events per raion

# Aggregate by raion (English name)
d_lvl2_ucdp_raion = d_lvl2_ucdp.groupby('raion_nominatim_en', as_index=False).agg(agg_dict_ucdp)

# Rename count column
d_lvl2_ucdp_raion = d_lvl2_ucdp_raion.rename(columns={'conflict_name': 'n_events'})

# Add oblast mapping (most common oblast per raion)
oblast_map_ucdp = d_lvl2_ucdp.groupby('raion_nominatim_en')['oblast'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan
)
d_lvl2_ucdp_raion['oblast'] = d_lvl2_ucdp_raion['raion_nominatim_en'].map(oblast_map_ucdp)

# Reorder columns: raion, oblast, n_events, deaths
col_order_ucdp_raion = ['raion_nominatim_en', 'oblast', 'n_events'] + ucdp_agg_vars
d_lvl2_ucdp_raion = d_lvl2_ucdp_raion[col_order_ucdp_raion]

# Sort by total events (descending)
d_lvl2_ucdp_raion = d_lvl2_ucdp_raion.sort_values('n_events', ascending=False).reset_index(drop=True)

# Summary
print(f"Raion-level aggregation: {len(d_lvl2_ucdp_raion)} raions")
print(f"Total events: {d_lvl2_ucdp_raion['n_events'].sum():,}")
print(f"Total deaths_b: {d_lvl2_ucdp_raion['deaths_b'].sum():,}")
print(f"Total deaths_civilians: {d_lvl2_ucdp_raion['deaths_civilians'].sum():,}")
print(f"\nTop 10 raions by event count:")
d_lvl2_ucdp_raion.head(10)

# Cell ID: lndorwhux7a

In [ ]:
# Export UCDP raion-level estimates for 1:n merge
d_lvl2_ucdp_raion.to_csv(f'{DATA_PROC}/d_lvl2_ucdp_raion.csv', index=False)

# Cell ID: p01qtg3owni

#### _Aggregate UCDP event counts by_ oblast

Collapses event-level UCDP data to _oblast_-level raw counts of Ukrainian military (`deaths_b`) and civilian deaths (`deaths_civilians`).

In [ ]:
# Build aggregation dict: sum deaths, count events
agg_dict_ucdp_obl = {var: 'sum' for var in ucdp_agg_vars}
agg_dict_ucdp_obl['conflict_name'] = 'count'

# Aggregate by oblast
d_lvl2_ucdp_oblast = d_lvl2_ucdp.groupby('oblast', as_index=False).agg(agg_dict_ucdp_obl)

# Rename count column
d_lvl2_ucdp_oblast = d_lvl2_ucdp_oblast.rename(columns={'conflict_name': 'n_events'})

# Add raion count per oblast
raion_counts_ucdp = d_lvl2_ucdp.groupby('oblast')['raion_nominatim_en'].nunique()
d_lvl2_ucdp_oblast['n_raions'] = d_lvl2_ucdp_oblast['oblast'].map(raion_counts_ucdp)

# Reorder columns: oblast, n_raions, n_events, deaths
col_order_ucdp_obl = ['oblast', 'n_raions', 'n_events'] + ucdp_agg_vars
d_lvl2_ucdp_oblast = d_lvl2_ucdp_oblast[col_order_ucdp_obl]

# Sort by total events (descending)
d_lvl2_ucdp_oblast = d_lvl2_ucdp_oblast.sort_values('n_events', ascending=False).reset_index(drop=True)

# Summary
print(f"Oblast-level aggregation: {len(d_lvl2_ucdp_oblast)} oblasts")
print(f"Total events: {d_lvl2_ucdp_oblast['n_events'].sum():,}")
print(f"Total deaths_b: {d_lvl2_ucdp_oblast['deaths_b'].sum():,}")
print(f"Total deaths_civilians: {d_lvl2_ucdp_oblast['deaths_civilians'].sum():,}")
print(f"\nOblast counts:")
d_lvl2_ucdp_oblast

# Cell ID: s7tqqfnlae

In [ ]:
# Export UCDP oblast-level estimates for 1:n merge
d_lvl2_ucdp_oblast.to_csv(f'{DATA_PROC}/d_lvl2_ucdp_oblast.csv', index=False)

# Cell ID: 8fnhcxia7mf